In [1]:
import os
import json
import time
import numpy as np
from google import genai
from google.genai import types
import chromadb
from rank_bm25 import BM25Okapi

# Initialize the modern Gemini Client
client = genai.Client()

# Initialize an in-memory Chroma DB client
chroma_client = chromadb.Client()

print("Setup complete. Libraries imported and Gemini client initialized.")

Setup complete. Libraries imported and Gemini client initialized.


In [5]:
# 1. Safely handle setting up the collection without triggering config reset guardrails
try:
    chroma_client.delete_collection("kb_collection_lab3_fixed")
except Exception:
    pass  # Collection didn't exist yet, which is fine

collection = chroma_client.get_or_create_collection(name="kb_collection_lab3_fixed")

# 2. Define standard embedding helper using the modern SDK
def get_embedding(text: str) -> list:
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
    )
    return [float(x) for x in response.embeddings[0].values]

# 3. Load knowledge base
with open("knowledge_base.json", "r", encoding="utf-8") as f:
    knowledge_base = json.load(f)


print(f"Indexing {len(knowledge_base)} passages into Chroma with explicit embeddings...")

# 4. Insert data explicitly providing text and vectors
for idx, item in enumerate(knowledge_base):
    passage_id = str(item.get("id"))
    source = item.get("source", "Unknown")
    text = item.get("text")
    
    print(f"[{idx+1}/{len(knowledge_base)}] Embedding & Indexing ID: {passage_id}...")
    
    vector_embedding = get_embedding(text)
    
    collection.add(
        documents=[text],
        embeddings=[vector_embedding],
        metadatas=[{"source": source}],
        ids=[passage_id]
    )
    
    # Rate limit protection: Stay safely under 15 RPM
    if idx < len(knowledge_base) - 1:
        time.sleep(4.5)

print("\nIndexing complete! Text and raw embeddings securely stored.")

Indexing 10 passages into Chroma with explicit embeddings...
[1/10] Embedding & Indexing ID: kb-01...
[2/10] Embedding & Indexing ID: kb-02...
[3/10] Embedding & Indexing ID: kb-03...
[4/10] Embedding & Indexing ID: kb-04...
[5/10] Embedding & Indexing ID: kb-05...
[6/10] Embedding & Indexing ID: kb-06...
[7/10] Embedding & Indexing ID: kb-07...
[8/10] Embedding & Indexing ID: kb-08...
[9/10] Embedding & Indexing ID: kb-09...
[10/10] Embedding & Indexing ID: kb-10...

Indexing complete! Text and raw embeddings securely stored.


In [6]:
# Tokenize the knowledge base documents for BM25 keyword matching
corpus_texts = [item["text"] for item in knowledge_base]
tokenized_corpus = [text.lower().split(" ") for text in corpus_texts]

# Initialize BM25 index
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 Keyword Index initialized successfully.")

BM25 Keyword Index initialized successfully.


In [19]:
# Evaluation dataset matching questions to their precise knowledge base IDs
eval_set = [
    {
        "question": "How long do I have to get a full refund?",
        "expected_id": "kb-04"
    },
    {
        "question": "How do I reset my password?",
        "expected_id": "kb-07"
    },
    {
        "question": "What should I do if I see error code 0x80070005 when saving?",
        "expected_id": "kb-08"
    },
    {
        "question": "Where is overnight parking permitted?",
        "expected_id": "kb-01"
    },
    {
        "question": "How do I stop monthly automatic billing?",
        "expected_id": "kb-05"
    }
]
print("Evaluation dataset successfully synced with the real database ground truth.")

Evaluation dataset successfully synced with the real database ground truth.


In [8]:
def retrieve_baseline(question: str) -> list:
    """Baseline Vector Retrieval using explicitly generated query embeddings."""
    query_vector = get_embedding(question)
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=3
    )
    
    if not results['ids'] or not results['ids'][0]:
        return []
        
    return list(zip(
        results['ids'][0], 
        results['documents'][0], 
        [m['source'] for m in results['metadatas'][0]]
    ))

In [9]:
def retrieve_hybrid(question: str) -> list:
    """Hybrid Search: Combines Explicit Vector Query and BM25 scores."""
    # 1. Get Dense Vector results
    query_vector = get_embedding(question)
    vector_res = collection.query(query_embeddings=[query_vector], n_results=3)
    
    combined_results = []
    seen_ids = set()
    
    if vector_res['ids'] and vector_res['ids'][0]:
        for v_id, doc, meta in zip(vector_res['ids'][0], vector_res['documents'][0], vector_res['metadatas'][0]):
            if v_id not in seen_ids:
                combined_results.append((v_id, doc, meta['source']))
                seen_ids.add(v_id)
    
    # 2. Get BM25 Keyword results
    tokenized_query = question.lower().split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)
    top_bm25_indices = np.argsort(bm25_scores)[::-1][:3]
    
    # 3. Merge unique BM25 results up to max capacity of 3
    for idx in top_bm25_indices:
        item = knowledge_base[idx]
        b_id = str(item["id"])
        if b_id not in seen_ids and len(combined_results) < 3:
            combined_results.append((b_id, item["text"], item.get("source", "Unknown")))
            seen_ids.add(b_id)
            
    return combined_results[:3]

In [10]:
def calculate_hit_rate(retrieved_docs: list, expected_id: str) -> int:
    """Returns 1 if expected_id is inside the retrieved documents, else 0."""
    retrieved_ids = [str(doc_tuple[0]) for doc_tuple in retrieved_docs]
    return 1 if str(expected_id) in retrieved_ids else 0

def generate_rag_answer(question: str, retrieved_docs: list) -> tuple:
    """Generates an answer using the retrieved context via 3.1-flash-lite."""
    context_str = ""
    for doc_id, doc_text, source in retrieved_docs:
        context_str += f"Source: {source}\nContent: {doc_text}\n\n"
        
    prompt = f"""Answer the user's question using ONLY the provided context. 
If the context doesn't contain the answer, say you don't know.

CONTEXT:
{context_str}

QUESTION: {question}
ANSWER:"""

    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=prompt,
    )
    return response.text.strip(), context_str

def judge_faithfulness(context: str, answer: str) -> str:
    """Uses Gemini-3.1-Flash-Lite as an evaluator to verify faithfulness (Yes/No)."""
    judge_prompt = f"""You are an unbiased quality control judge. 
Review the following CONTEXT and the corresponding ANSWER.
Determine if the ANSWER is fully supported by the facts inside the CONTEXT. 
If the answer contains any outside facts or unmentioned assumptions, reply with 'No'. 
If it is completely faithful to the context, reply with 'Yes'.

CONTEXT:
{context}

ANSWER:
{answer}

Your response must be exactly one word, either 'Yes' or 'No'."""

    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=judge_prompt,
    )
    result = response.text.strip().lower()
    return "Yes" if "yes" in result else "No"

In [20]:
baseline_hits = 0
baseline_faithfuls = 0
hybrid_hits = 0
hybrid_faithfuls = 0

print("Starting evaluation across both setups...\n")

for idx, eval_item in enumerate(eval_set):
    q = eval_item["question"]
    expected = eval_item["expected_id"]
    
    print(f"Evaluating Q{idx+1}: '{q}'")
    
    # --- EVALUATE BASELINE ---
    baseline_docs = retrieve_baseline(q)
    b_hit = calculate_hit_rate(baseline_docs, expected)
    baseline_hits += b_hit
    
    b_ans, b_ctx = generate_rag_answer(q, baseline_docs)
    b_faith = judge_faithfulness(b_ctx, b_ans)
    if b_faith == "Yes": baseline_faithfuls += 1
    
    # --- EVALUATE HYBRID ---
    hybrid_docs = retrieve_hybrid(q)
    h_hit = calculate_hit_rate(hybrid_docs, expected)
    hybrid_hits += h_hit
    
    h_ans, h_ctx = generate_rag_answer(q, hybrid_docs)
    h_faith = judge_faithfulness(h_ctx, h_ans)
    if h_faith == "Yes": hybrid_faithfuls += 1
    
    # Quota safety sleep between context evaluation runs
    time.sleep(2.5)

print("\nEvaluation Complete!")

Starting evaluation across both setups...

Evaluating Q1: 'How long do I have to get a full refund?'
Evaluating Q2: 'How do I reset my password?'
Evaluating Q3: 'What should I do if I see error code 0x80070005 when saving?'
Evaluating Q4: 'Where is overnight parking permitted?'
Evaluating Q5: 'How do I stop monthly automatic billing?'

Evaluation Complete!


In [23]:
total_qs = len(eval_set)

baseline_hit_rate = (baseline_hits / total_qs) * 100
baseline_faith_rate = (baseline_faithfuls / total_qs) * 100
hybrid_hit_rate = (hybrid_hits / total_qs) * 100
hybrid_faith_rate = (hybrid_faithfuls / total_qs) * 100

print("### Copy the Markdown below for your eval_results.md file:\n")
print("---")
print("## Lab 3 Evaluation Results Summary Table\n")
print(f"| Metric | Baseline (Dense Vector) | Upgraded (Hybrid Search) |")
print(f"| --- | --- | --- |")
print(f"| **Retrieval Hit Rate** | {baseline_hit_rate:.1f}% ({baseline_hits}/{total_qs}) | {hybrid_hit_rate:.1f}% ({hybrid_hits}/{total_qs}) |")
print(f"| **Faithfulness Score** | {baseline_faith_rate:.1f}% ({baseline_faithfuls}/{total_qs}) | {hybrid_faith_rate:.1f}% ({hybrid_faithfuls}/{total_qs}) |")


### Copy the Markdown below for your eval_results.md file:

---
## Lab 3 Evaluation Results Summary Table

| Metric | Baseline (Dense Vector) | Upgraded (Hybrid Search) |
| --- | --- | --- |
| **Retrieval Hit Rate** | 100.0% (5/5) | 100.0% (5/5) |
| **Faithfulness Score** | 80.0% (4/5) | 80.0% (4/5) |
